# Task 2: Pair-Level Sentiment Classification

This notebook implements **Task 2: sentiment classification** for our cannabis project.

## Goal
Predict the sentiment for a **(text, product)** pair, where:

- product ∈ {`flower`, `oil`, `gummies`, `vape`, `topical`}
- sentiment ∈ {`positive`, `negative`, `neutral`}

## Key design choices
- **Merge pair-level labels with `cleaned_dataset.csv`** to recover cleaned text.
- Build **product-conditioned inputs** so the same text can yield different sentiment labels for different product forms.
- **Keep emojis for the transformer model**.
- **Demojize text for classical models** so emoji sentiment is converted into lexical tokens.
- **Split by `text_id`** to avoid leakage across train/validation/test.
- Compare:
  - Classical models on **demojized text**
  - DistilBERT on **original emoji-preserved text**

Example:

Text:
> "Oil helps me sleep, but vape makes me cough."

Pair rows:
- `(text, oil) -> positive`
- `(text, vape) -> negative`

## What this notebook includes
1. Load and validate the pair-level dataset
2. Filter to the final 5 product labels
3. Split train/val/test by `text_id` to avoid leakage
4. Build product-conditioned text inputs
5. Train classical models:
   - TF-IDF + Logistic Regression
   - TF-IDF + LinearSVC
6. Train a transformer model:
   - DistilBERT for 3-class classification
7. Evaluate:
   - accuracy
   - macro F1
   - weighted F1
   - per-class precision/recall/F1
   - confusion matrix
   - multiclass AUROC (if probabilities available)
   - ECE / calibration
8. Save predictions, metrics, and figures


In [ ]:
# =========================
# 1. Imports
# =========================
import os
import re
import random
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)

import emoji

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# =========================
# 2. Paths
# =========================
PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis")
PAIR_PATH = PROJECT_ROOT / "data_processing" / "data" / "final" / "pair_level_gpt41mini_clean.csv"
CLEAN_PATH = PROJECT_ROOT / "data_processing" / "data" / "processed" / "cleaned_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "Modeling" / "task2_sentiment_fast_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(PAIR_PATH.exists(), CLEAN_PATH.exists(), OUTPUT_DIR)

In [ ]:
# =========================
# 3. Load data
# =========================
pair_df = pd.read_csv(PAIR_PATH)
clean_df = pd.read_csv(CLEAN_PATH)

print(pair_df.shape)
print(clean_df.shape)
print(pair_df.columns.tolist())
print(clean_df.columns.tolist())

In [ ]:
# =========================
# 4. Helper functions
# =========================
def find_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find any of columns: {candidates}")
    return None

def remove_extra_spaces(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def demojize_text(x):
    return emoji.demojize(str(x), delimiters=(" ", " "))

def basic_clean(x):
    x = str(x)
    x = re.sub(r"http\S+|www\.\S+", " ", x)
    x = remove_extra_spaces(x)
    return x

In [ ]:
# =========================
# 5. Merge pair-level labels back to cleaned text
# =========================
pair_id_col = find_col(pair_df, ["text_id", "id", "ext_id"])
product_col = find_col(pair_df, ["product"])
sentiment_col = find_col(pair_df, ["sentiment"])
date_col = find_col(pair_df, ["date_utc"], required=False)

clean_id_col = find_col(clean_df, ["text_id", "id", "ext_id", "record_id", "doc_id"])
clean_text_col = find_col(clean_df, ["text", "clean_text", "full_text", "body", "content"])

pair_small = pair_df[[pair_id_col, product_col, sentiment_col] + ([date_col] if date_col else [])].copy()
pair_small = pair_small.rename(columns={
    pair_id_col: "text_id",
    product_col: "product",
    sentiment_col: "sentiment",
    **({date_col: "date_utc"} if date_col else {})
})

clean_small = clean_df[[clean_id_col, clean_text_col]].copy()
clean_small = clean_small.rename(columns={
    clean_id_col: "merge_id",
    clean_text_col: "text"
})

pair_small["text_id"] = pair_small["text_id"].astype(str)
clean_small["merge_id"] = clean_small["merge_id"].astype(str)

merged_df = pair_small.merge(
    clean_small,
    left_on="text_id",
    right_on="merge_id",
    how="left"
)

print("merged_df:", merged_df.shape)
print("missing text:", merged_df["text"].isna().sum())
merged_df.head()

In [ ]:
# =========================
# 6. Filter to final products and sentiments
# =========================
FINAL_PRODUCTS = ["flower", "oil", "gummies", "vape", "topical"]
FINAL_SENTIMENTS = ["negative", "neutral", "positive"]

df = merged_df.copy()
df["product"] = df["product"].astype(str).str.lower().str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df["text"] = df["text"].fillna("").astype(str).map(basic_clean)

df = df[df["product"].isin(FINAL_PRODUCTS)].copy()
df = df[df["sentiment"].isin(FINAL_SENTIMENTS)].copy()
df = df[df["text"] != ""].copy()

print(df.shape)
df.head()

In [1]:
# =========================
# 7. Product-conditioned text
# =========================
df["text_classical"] = (
    "[PRODUCT=" + df["product"].astype(str) + "] " + df["text"].astype(str)
).map(demojize_text).map(remove_extra_spaces)

df["text_transformer"] = (
    "[PRODUCT=" + df["product"].astype(str) + "] " + df["text"].astype(str)
).map(remove_extra_spaces)

label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["y"] = df["sentiment"].map(label_map).astype(int)

df[["product", "sentiment", "text_classical", "text_transformer"]].head()

NameError: name 'df' is not defined

## Split by `text_id` to avoid leakage

This is critical.

If the same original text appears in both train and test through different product rows, the evaluation will be overly optimistic.

So we split **unique `text_id`s first**, then bring along all pair rows for those IDs.


In [ ]:
# =========================
# 8. Split by text_id to avoid leakage
# =========================
unique_ids = df["text_id"].astype(str).unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.15, random_state=SEED)
train_ids, val_ids = train_test_split(train_ids, test_size=0.17647, random_state=SEED)

train_df = df[df["text_id"].astype(str).isin(train_ids)].copy()
val_df = df[df["text_id"].astype(str).isin(val_ids)].copy()
test_df = df[df["text_id"].astype(str).isin(test_ids)].copy()

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["sentiment"].value_counts())

## Classical Models: 

A strong and interpretable baseline for multiclass text classification.


In [ ]:
# =========================
# 9. Classical baselines
# =========================
X_train = train_df["text_classical"].tolist()
X_val = val_df["text_classical"].tolist()
X_test = test_df["text_classical"].tolist()

y_train = train_df["y"].values
y_val = val_df["y"].values
y_test = test_df["y"].values

results = []

def evaluate_multiclass(y_true, y_pred, model_name):
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })
    print("\n", model_name)
    print("accuracy:", round(acc, 4))
    print("macro_f1:", round(macro_f1, 4))
    print("weighted_f1:", round(weighted_f1, 4))
    print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"], zero_division=0))

In [ ]:
# CountVectorizer + LinearSVC
count_vec = CountVectorizer(lowercase=True, stop_words="english", min_df=2, max_df=0.98, ngram_range=(1,2))
Xtr_count = count_vec.fit_transform(X_train)
Xte_count = count_vec.transform(X_test)

count_svc = LinearSVC(class_weight="balanced")
count_svc.fit(Xtr_count, y_train)

test_pred = count_svc.predict(Xte_count)
evaluate_multiclass(y_test, test_pred, "CountVectorizer + LinearSVC")

In [ ]:
# TF-IDF + Logistic Regression
tfidf_vec = TfidfVectorizer(lowercase=True, stop_words="english", min_df=2, max_df=0.98, ngram_range=(1,2), sublinear_tf=True)
Xtr_tfidf = tfidf_vec.fit_transform(X_train)
Xte_tfidf = tfidf_vec.transform(X_test)

tfidf_logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
tfidf_logreg.fit(Xtr_tfidf, y_train)

test_pred = tfidf_logreg.predict(Xte_tfidf)
evaluate_multiclass(y_test, test_pred, "TF-IDF + LogisticRegression")

In [ ]:
# Character n-grams + Logistic Regression
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_df=0.99, sublinear_tf=True)
Xtr_char = char_vec.fit_transform(X_train)
Xte_char = char_vec.transform(X_test)

char_logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
char_logreg.fit(Xtr_char, y_train)

test_pred = char_logreg.predict(Xte_char)
evaluate_multiclass(y_test, test_pred, "Char n-grams + LogisticRegression")

In [ ]:
classical_results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
classical_results_df

## Transformer Model: prajjwal1/bert-tiny

This model is optimized for speed, especially on CPU:

- only 1-2 epochs
- `max_len=96`
- optional subset for quick sanity checks
- freezes encoder by default, so only the classification head trains

In [ ]:
# =========================
# 10. transformer configuration
# =========================
FAST_MODEL_NAME = "prajjwal1/bert-tiny"
# FAST_MODEL_NAME = "distilbert-base-uncased"

MAX_LEN = 96
BATCH_SIZE = 16 if torch.cuda.is_available() else 8
EPOCHS = 1 if not torch.cuda.is_available() else 2
LR = 2e-4
FREEZE_ENCODER = True

USE_SUBSET = False
SUBSET_N = 12000

print(FAST_MODEL_NAME, MAX_LEN, BATCH_SIZE, EPOCHS, FREEZE_ENCODER)

In [ ]:
# =========================
# 11. Optional subset to make the run much faster
# =========================
if USE_SUBSET:
    train_df_fast = train_df.sample(min(SUBSET_N, len(train_df)), random_state=SEED).copy()
    val_df_fast = val_df.sample(min(max(2000, SUBSET_N // 5), len(val_df)), random_state=SEED).copy()
    test_df_fast = test_df.sample(min(max(2000, SUBSET_N // 5), len(test_df)), random_state=SEED).copy()
else:
    train_df_fast = train_df.copy()
    val_df_fast = val_df.copy()
    test_df_fast = test_df.copy()

print(train_df_fast.shape, val_df_fast.shape, test_df_fast.shape)

In [ ]:
# =========================
# 12. Torch dataset
# =========================
tokenizer = AutoTokenizer.from_pretrained(FAST_MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, df, text_col="text_transformer", label_col="y", max_len=96):
        self.texts = df[text_col].fillna("").astype(str).tolist()
        self.labels = df[label_col].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = SentimentDataset(train_df_fast, max_len=MAX_LEN)
val_ds = SentimentDataset(val_df_fast, max_len=MAX_LEN)
test_ds = SentimentDataset(test_df_fast, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# =========================
# 13. Fast classifier model
# =========================
class FastSentimentClassifier(nn.Module):
    def __init__(self, model_name, num_labels=3, freeze_encoder=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, num_labels)

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

model = FastSentimentClassifier(FAST_MODEL_NAME, num_labels=3, freeze_encoder=FREEZE_ENCODER).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

print("total params:", sum(p.numel() for p in model.parameters()))
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# =========================
# 14. Train / evaluate loops
# =========================
def run_eval(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader))
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, macro_f1, weighted_f1, np.array(all_labels), np.array(all_preds)

best_state = None
best_val_macro = -1

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / max(1, len(train_loader))
    val_loss, val_acc, val_macro, val_weighted, _, _ = run_eval(model, val_loader)

    print({
        "epoch": epoch + 1,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_acc": round(val_acc, 4),
        "val_macro_f1": round(val_macro, 4),
        "val_weighted_f1": round(val_weighted, 4)
    })

    if val_macro > best_val_macro:
        best_val_macro = val_macro
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)

In [ ]:
# =========================
# 15. Final transformer test evaluation
# =========================
test_loss, test_acc, test_macro, test_weighted, y_true_tf, y_pred_tf = run_eval(model, test_loader)

transformer_result = pd.DataFrame([{
    "model": f"{FAST_MODEL_NAME} (fast)",
    "accuracy": test_acc,
    "macro_f1": test_macro,
    "weighted_f1": test_weighted
}])

print(classification_report(y_true_tf, y_pred_tf, target_names=["negative", "neutral", "positive"], zero_division=0))
transformer_result

In [ ]:
# =========================
# 16. Final comparison table
# =========================
final_comparison = pd.concat([classical_results_df, transformer_result], ignore_index=True)
final_comparison = final_comparison.sort_values("macro_f1", ascending=False).reset_index(drop=True)
final_comparison.to_csv(OUTPUT_DIR / "final_model_comparison.csv", index=False)
final_comparison